# Clase 174 — Entrenamiento a escala con Vertex AI

Cierre del bloque de despliegue: lanzar **training jobs a escala** en Vertex AI (cluster
automático, GPUs/TPUs on-demand, HP tuning con Vizier). Alternativas: SageMaker Training,
Azure ML Jobs, Modal, Together, RunPod.

Requiere: `google-cloud-aiplatform` (opcional). Clase de despliegue: el código es correcto pero
no se ejecuta (necesita cuenta GCP).

## 1. Empaquetar el training script en un container

```dockerfile
FROM us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-15:latest
WORKDIR /app
COPY train.py .
ENTRYPOINT ["python", "train.py"]
```
```bash
docker build -t us-central1-docker.pkg.dev/PROY/repo/train:v1 .
docker push us-central1-docker.pkg.dev/PROY/repo/train:v1
```

## 2. Lanzar un Custom Training Job

In [ ]:
try:
    from google.cloud import aiplatform
    GCP_OK = True
except Exception:
    GCP_OK = False
    print("google-cloud-aiplatform no instalado -> se muestra la API (no se ejecuta)")

IMAGE = "us-central1-docker.pkg.dev/PROY/repo/train:v1"
worker_pool_specs = [{
    "machine_spec": {"machine_type": "n1-standard-8",
                     "accelerator_type": "NVIDIA_TESLA_T4",
                     "accelerator_count": 1},
    "replica_count": 1,
    "container_spec": {"image_uri": IMAGE},
}]

if GCP_OK:
    aiplatform.init(project="mi-proyecto", location="us-central1")
    job = aiplatform.CustomJob(display_name="fashion-exp1",
                               worker_pool_specs=worker_pool_specs)
    job.run(sync=False)
    print("job lanzado:", job.display_name)
else:
    print("aiplatform.CustomJob(display_name=..., worker_pool_specs=[...]).run()")

## 3. Hyperparameter tuning con Vizier

Vizier es el optimizador black-box (búsqueda bayesiana) integrado. Se define el espacio de
búsqueda y la métrica a maximizar.

In [ ]:
if GCP_OK:
    from google.cloud.aiplatform import hyperparameter_tuning as hpt
    hp_job = aiplatform.HyperparameterTuningJob(
        display_name="fashion-hpo",
        custom_job=job,
        metric_spec={"accuracy": "maximize"},
        parameter_spec={
            "learning_rate": hpt.DoubleParameterSpec(min=1e-4, max=1e-2, scale="log"),
            "batch_size":    hpt.DiscreteParameterSpec(values=[32, 64, 128], scale="linear"),
        },
        max_trial_count=20,
        parallel_trial_count=4,
    )
    hp_job.run()
    print("HP tuning con Vizier: 20 trials, 4 en paralelo")
else:
    print("HyperparameterTuningJob(metric_spec={'accuracy':'maximize'},")
    print("    parameter_spec={'learning_rate': DoubleParameterSpec(1e-4, 1e-2, 'log')}, ...)")

## 4. Multi-GPU / TPU spec

Para escalar el job basta cambiar el `machine_spec`:

- `machine_type='a2-highgpu-4g'` → 4× A100.
- TPU pods para transformers/LLMs muy grandes.

Y usar **spot/preemptible** (~70% más barato) con checkpointing frecuente para tolerar
interrupciones.

In [ ]:
multi_gpu_spec = [{
    "machine_spec": {"machine_type": "a2-highgpu-4g",
                     "accelerator_type": "NVIDIA_TESLA_A100",
                     "accelerator_count": 4},
    "replica_count": 1,
    "container_spec": {"image_uri": IMAGE},
}]
print("machine_type:", multi_gpu_spec[0]["machine_spec"]["machine_type"], "-> 4x A100")
print("tip: checkpointing frecuente para usar spot/preemptible (~70% mas barato)")

## 5. Vertex vs local y alternativas

| Usar cloud cuando | Alternativas |
|---|---|
| dataset no entra en RAM local | SageMaker Training (AWS) |
| training > 1 día | Azure ML Jobs |
| HP tuning con muchos trials | Modal / RunPod (DX, pay-per-second) |
| multi-GPU / TPU | Together AI (LLMs open-source) |

Local para iterar; cloud para escalar. TPU brilla en TF/JAX (LLMs, ViT); GPU es más universal.

## 6. Escribir el Dockerfile del training a disco

In [ ]:
dockerfile = '''FROM us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-15:latest
WORKDIR /app
COPY train.py .
ENTRYPOINT ["python", "train.py"]
'''
with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile)
print("Dockerfile escrito. Build & push:")
print("  docker build -t us-central1-docker.pkg.dev/PROY/repo/train:v1 .")
print("  docker push  us-central1-docker.pkg.dev/PROY/repo/train:v1")

## 7. Monitoreo con Vertex TensorBoard

In [ ]:
if GCP_OK:
    tb = aiplatform.Tensorboard.create(display_name="fashion-tb")
    # job.run(tensorboard=tb.resource_name, service_account="...")
    print("Vertex TensorBoard:", tb.resource_name)
else:
    print("aiplatform.Tensorboard.create(display_name='fashion-tb')")
    print("job.run(tensorboard=tb.resource_name)  # streamea metricas del cluster")

## Ejercicios

1. Escribir un `Dockerfile` con TF + `train.py`, construirlo y subirlo a Artifact Registry.
2. Lanzar un `CustomJob` de 1 GPU (T4) con logs y modelo de salida a GCS.
3. Configurar un `HyperparameterTuningJob` con Vizier (20 trials) sobre `learning_rate` y `batch_size`.
4. Verificar que el modelo entrenado en Vertex logra accuracy ≥ 0.87 y queda guardado en GCS.

## Conclusiones

- Vertex AI lanza training jobs a escala empaquetando el script en un container.
- El `worker_pool_specs` define el cluster (machine_type, accelerator, réplicas).
- Vizier hace HP tuning bayesiano con múltiples trials en paralelo.
- Spot/preemptible + checkpointing abaratan ~70%; local para iterar, cloud para escalar.